# QM640 Food Price Affordability AI
## Notebook 01 - Synopsis-Aligned Data Acquisition, Audit, and Integration

**Student:** Piyush Soni  
**Environment:** Google Colab  
**Controlling design:** Original approved synopsis  
**Study period:** Maximum official history through 2026 YTD, subject to source availability

This notebook preserves the original RQ1-RQ4 variable requirements while
reproducing the tested acquisition pipeline. It creates current modeling panels,
audits every confirmatory data source, generates input-schema templates, and
prints missing-data actions immediately.


## 1. Runtime configuration

Google Colab already includes pandas and NumPy, so no package installation is
needed. Avoiding unnecessary installations makes execution faster and more
reliable.

Set `SAVE_TO_DRIVE = True` to keep results after the Colab session ends. Google
Drive will ask for authorization once.


In [3]:
# Standard-library imports handle files, Git, JSON, and environment detection.
from pathlib import Path
import json
import os
import subprocess
import sys
import time
import warnings

# pandas provides efficient tabular operations; NumPy provides vectorized arithmetic.
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Change this to False if you only want temporary files inside the Colab session.
SAVE_TO_DRIVE = True

REPO_URL = "https://github.com/piyushsoni88/qm640-food-affordability-ai.git"
REPO_NAME = "qm640-food-affordability-ai"

# Import detection is more reliable than checking sys.modules because Colab may
# not preload google.colab before the first user cell runs.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in Google Colab: {IN_COLAB}")
print(f"pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


Running in Google Colab: True
pandas version: 2.2.2
NumPy version: 2.0.2


## 2. Connect Google Drive and obtain the repository

The repository is cloned with `--depth 1` because earlier Git history is not
needed for analysis. This reduces download time and temporary storage. If the
repository already exists in the current Colab runtime, it is updated instead.


In [4]:
# Mount Drive only when requested and when the notebook is running in Colab.
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

# Colab uses /content for fast temporary computation. When run locally, this cell
# detects the repository from the current working directory instead.
if IN_COLAB:
    REPO_ROOT = Path("/content") / REPO_NAME
    if not REPO_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)],
            check=True,
        )
    else:
        # A fast-forward-only pull avoids accidental merge commits in a notebook.
        subprocess.run(
            ["git", "-C", str(REPO_ROOT), "pull", "--ff-only"],
            check=True,
        )
else:
    candidate = Path.cwd().resolve()
    REPO_ROOT = candidate.parent if candidate.name == "notebooks" else candidate
    if not (REPO_ROOT / "data" / "curated").exists():
        raise FileNotFoundError(
            "Run this notebook from the repository root or its notebooks folder."
        )

CURATED = REPO_ROOT / "data" / "curated"

# Persist outputs in Drive, while using /content for the cloned input repository.
if IN_COLAB and SAVE_TO_DRIVE:
    OUTPUT_ROOT = Path("/content/drive/MyDrive/QM640_Food_Affordability")
else:
    # The optional override is useful for non-Colab validation and automated tests.
    OUTPUT_ROOT = Path(os.environ.get("QM640_OUTPUT_ROOT", str(REPO_ROOT)))

PROCESSED = OUTPUT_ROOT / "data" / "processed"
REPORT_OUTPUT = OUTPUT_ROOT / "reports" / "notebook_outputs"
PROCESSED.mkdir(parents=True, exist_ok=True)
REPORT_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f"Input repository: {REPO_ROOT}")
print(f"Curated inputs:   {CURATED}")
print(f"Saved outputs:    {OUTPUT_ROOT}")


Mounted at /content/drive
Input repository: /content/qm640-food-affordability-ai
Curated inputs:   /content/qm640-food-affordability-ai/data/curated
Saved outputs:    /content/drive/MyDrive/QM640_Food_Affordability


## 3. Audit all curated input files

Each CSV is read in chunks. Chunking caps peak memory rather than loading every
dataset at once. The resulting inventory supplies evidence for the interim report
and detects accidentally missing or empty files.


In [5]:
def audit_csv_gz(path: Path, chunksize: int = 250_000) -> dict:
    """Return actual rows/columns for one compressed CSV with bounded memory."""
    row_count = 0
    columns = None
    for chunk in pd.read_csv(path, chunksize=chunksize, low_memory=False):
        row_count += len(chunk)
        if columns is None:
            columns = list(chunk.columns)
    return {
        "file": path.name,
        "rows": row_count,
        "columns": len(columns or []),
        "compressed_bytes": path.stat().st_size,
        "compressed_mb": path.stat().st_size / 1_000_000,
    }


started = time.perf_counter()
curated_files = sorted(CURATED.glob("*.csv.gz"))
if not curated_files:
    raise FileNotFoundError(f"No curated .csv.gz files were found in {CURATED}")

inventory = pd.DataFrame([audit_csv_gz(path) for path in curated_files])
inventory = inventory.sort_values("rows", ascending=False).reset_index(drop=True)
inventory.to_csv(PROCESSED / "source_inventory.csv", index=False)

print(inventory.to_string(index=False, formatters={"compressed_mb": "{:.2f}".format}))
print(f"\nFiles audited: {len(inventory)}")
print(f"Curated analytical rows: {inventory['rows'].sum():,}")
print(f"Compressed input size: {inventory['compressed_mb'].sum():,.2f} MB")
print(f"Audit elapsed time: {time.perf_counter() - started:,.1f} seconds")


                                                           file   rows  columns  compressed_bytes compressed_mb
agmarknet_historical_8_commodities_daily_state_2000_2026.csv.gz 969377       10          12268568         12.27
     nasa_power_india_all_states_uts_daily_2000_2026_ytd.csv.gz 349272       12           5222965          5.22
       india_food_affordability_panel_36x8_2000_2026_ytd.csv.gz  91872       39           2111706          2.11
           india_food_affordability_panel_15x8_2005_2025.csv.gz  30240       30            606724          0.61
                           faostat_india_crop_production.csv.gz  26432       14            205759          0.21
   nasa_power_india_all_states_uts_monthly_2000_2026_ytd.csv.gz  11484       16            313945          0.31
                           faostat_india_producer_prices.csv.gz   5665       15             39922          0.04
            agmarknet_official_daily_snapshot_2026-07-28.csv.gz   3824       11             63198       

## 4. Efficiently aggregate historical AGMARKNET prices

The curated AGMARKNET file contains daily state–commodity aggregates representing
the much larger official raw dataset. We use:

- `usecols` to avoid loading unused columns;
- explicit dtypes to reduce memory;
- `chunksize` to bound peak RAM;
- vectorized weighted-price calculation; and
- two-stage aggregation, which is mathematically equivalent to a full group-by.

Weights are `source_rows`, so a daily mean based on many official observations
contributes more than a daily mean based on one observation.


In [6]:
AGMARKNET_FILE = (
    CURATED / "agmarknet_historical_8_commodities_daily_state_2000_2026.csv.gz"
)
AG_COLS = [
    "date",
    "State",
    "Commodity",
    "source_rows",
    "mean_modal_price_rs_per_quintal",
]

# Legacy labels are mapped to the labels used by the 36-state/UT climate panel.
# This prevents avoidable join failures without changing the underlying geography.
STATE_NAME_MAP = {
    "Andaman and Nicobar": "Andaman and Nicobar Islands",
    "Chattisgarh": "Chhattisgarh",
    "Keralam": "Kerala",
    "NCT of Delhi": "Delhi",
    "Uttrakhand": "Uttarakhand",
}

partial_aggregates = []
daily_aggregate_rows = 0
underlying_source_rows = 0
started = time.perf_counter()

reader = pd.read_csv(
    AGMARKNET_FILE,
    usecols=AG_COLS,
    dtype={
        "State": "category",
        "Commodity": "category",
        "source_rows": "int64",
        "mean_modal_price_rs_per_quintal": "float64",
    },
    parse_dates=["date"],
    chunksize=250_000,
)

for part_number, chunk in enumerate(reader, start=1):
    daily_aggregate_rows += len(chunk)
    underlying_source_rows += int(chunk["source_rows"].sum())

    # Convert to strings before replacement because the mapped labels introduce
    # values that are not necessarily present in the original categorical dtype.
    chunk["region"] = chunk["State"].astype(str).replace(STATE_NAME_MAP)
    chunk["date"] = chunk["date"].dt.to_period("M").dt.to_timestamp()

    # Weighted sums let us combine chunks exactly in a second aggregation pass.
    chunk["weighted_price"] = (
        chunk["mean_modal_price_rs_per_quintal"] * chunk["source_rows"]
    )
    grouped = (
        chunk.groupby(
            ["date", "region", "Commodity"],
            as_index=False,
            observed=True,
            sort=False,
        )
        .agg(
            weighted_price=("weighted_price", "sum"),
            source_rows=("source_rows", "sum"),
            daily_records=("State", "size"),
        )
    )
    partial_aggregates.append(grouped)
    print(f"Processed chunk {part_number}: {len(chunk):,} rows")

# The second pass combines groups that occurred in different input chunks.
monthly_prices = (
    pd.concat(partial_aggregates, ignore_index=True)
    .groupby(
        ["date", "region", "Commodity"],
        as_index=False,
        observed=True,
        sort=False,
    )
    .agg(
        weighted_price=("weighted_price", "sum"),
        source_rows=("source_rows", "sum"),
        daily_records=("daily_records", "sum"),
    )
)
monthly_prices["modal_price_rs_per_quintal"] = (
    monthly_prices["weighted_price"] / monthly_prices["source_rows"]
)
monthly_prices = monthly_prices.drop(columns="weighted_price").sort_values(
    ["date", "region", "Commodity"]
)

monthly_price_path = PROCESSED / "agmarknet_monthly_state_commodity.csv.gz"
monthly_prices.to_csv(monthly_price_path, index=False, compression="gzip")

print("\nAGMARKNET aggregation results")
print(f"Daily aggregate rows read: {daily_aggregate_rows:,}")
print(f"Underlying official observations: {underlying_source_rows:,}")
print(f"Monthly state–commodity rows: {len(monthly_prices):,}")
print(f"States/UT labels: {monthly_prices['region'].nunique()}")
print(f"Commodities: {monthly_prices['Commodity'].nunique()}")
print(f"Date range: {monthly_prices['date'].min().date()} to {monthly_prices['date'].max().date()}")
print(f"Elapsed time: {time.perf_counter() - started:,.1f} seconds")
print(f"Saved: {monthly_price_path}")


Processed chunk 1: 250,000 rows
Processed chunk 2: 250,000 rows
Processed chunk 3: 250,000 rows
Processed chunk 4: 219,377 rows

AGMARKNET aggregation results
Daily aggregate rows read: 969,377
Underlying official observations: 18,836,462
Monthly state–commodity rows: 41,792
States/UT labels: 32
Commodities: 8
Date range: 2001-01-01 to 2026-07-01
Elapsed time: 6.4 seconds
Saved: /content/drive/MyDrive/QM640_Food_Affordability/data/processed/agmarknet_monthly_state_commodity.csv.gz


## 5. Validate the AGMARKNET aggregation

Assertions intentionally stop execution if key totals, uniqueness, or price rules
fail. A failed assertion is evidence that the source or transformation changed
and should be investigated before modeling.


In [7]:
# The known source total comes from the downloaded official historical snapshot.
EXPECTED_UNDERLYING_OBSERVATIONS = 18_836_462

validation = {
    "underlying_total_matches_snapshot": (
        underlying_source_rows == EXPECTED_UNDERLYING_OBSERVATIONS
    ),
    "source_rows_preserved_after_aggregation": (
        int(monthly_prices["source_rows"].sum()) == underlying_source_rows
    ),
    "unique_state_month_commodity_keys": (
        not monthly_prices.duplicated(["date", "region", "Commodity"]).any()
    ),
    "all_prices_positive": (
        monthly_prices["modal_price_rs_per_quintal"].gt(0).all()
    ),
    "all_weights_positive": monthly_prices["source_rows"].gt(0).all(),
}

for check, passed in validation.items():
    print(f"{'PASS' if passed else 'FAIL'} — {check}")

if not all(validation.values()):
    raise AssertionError("One or more AGMARKNET validation checks failed.")

commodity_coverage = (
    monthly_prices.groupby("Commodity", observed=True, as_index=False)
    .agg(
        monthly_rows=("date", "size"),
        states_uts=("region", "nunique"),
        start_date=("date", "min"),
        end_date=("date", "max"),
        underlying_observations=("source_rows", "sum"),
    )
    .sort_values("underlying_observations", ascending=False)
)
commodity_coverage.to_csv(REPORT_OUTPUT / "01_commodity_coverage.csv", index=False)
print("\nCommodity coverage")
print(commodity_coverage.to_string(index=False))


PASS — underlying_total_matches_snapshot
PASS — source_rows_preserved_after_aggregation
PASS — unique_state_month_commodity_keys
PASS — all_prices_positive
PASS — all_weights_positive

Commodity coverage
                  Commodity  monthly_rows  states_uts start_date   end_date  underlying_observations
                      Wheat          4853          27 2001-04-01 2026-07-01                  3470626
                     Potato          7297          32 2001-01-01 2026-07-01                  3406949
                      Onion          7039          31 2001-01-01 2026-07-01                  3338857
                     Tomato          6637          31 2001-02-01 2026-07-01                  2968850
                       Rice          4784          28 2001-03-01 2026-07-01                  1867046
   Bengal Gram(Gram)(Whole)          4445          23 2001-01-01 2026-07-01                  1534906
                   Soyabean          2783          24 2001-03-01 2026-07-01              

## 6. Join monthly prices to NASA POWER climate data

This is a many-to-one join: each state–commodity–month price record receives one
state/UT-month climate record. `validate="many_to_one"` makes pandas raise an
error if the climate keys are unexpectedly duplicated.

Climate values are representative administrative-capital points, not
area-weighted state averages; this limitation must remain in the report.


In [8]:
CLIMATE_FILE = (
    CURATED / "nasa_power_india_all_states_uts_monthly_2000_2026_ytd.csv.gz"
)
climate_columns = [
    "region",
    "admin_type",
    "reference_location",
    "spatial_representation",
    "date",
    "rainfall_mm",
    "temperature_c",
    "temperature_max_c",
    "temperature_min_c",
    "relative_humidity_pct",
    "observed_days",
    "expected_days",
    "is_partial_month",
    "data_status",
]
climate = pd.read_csv(
    CLIMATE_FILE,
    usecols=climate_columns,
    parse_dates=["date"],
)

if climate.duplicated(["region", "date"]).any():
    raise AssertionError("Climate data contains duplicate region-date keys.")

state_modeling_panel = monthly_prices.merge(
    climate,
    on=["region", "date"],
    how="left",
    validate="many_to_one",
    indicator=True,
)
climate_match_rate = state_modeling_panel["_merge"].eq("both").mean()
unmatched_regions = sorted(
    state_modeling_panel.loc[
        state_modeling_panel["_merge"].ne("both"), "region"
    ].unique()
)
state_modeling_panel = state_modeling_panel.drop(columns="_merge")

state_panel_path = PROCESSED / "modeling_state_monthly.csv.gz"
state_modeling_panel.to_csv(state_panel_path, index=False, compression="gzip")

print(f"State modeling rows: {len(state_modeling_panel):,}")
print(f"Climate match rate: {climate_match_rate:.2%}")
print(f"Unmatched price-region labels: {unmatched_regions or 'None'}")
print(f"Saved: {state_panel_path}")


State modeling rows: 41,792
Climate match rate: 100.00%
Unmatched price-region labels: None
Saved: /content/drive/MyDrive/QM640_Food_Affordability/data/processed/modeling_state_monthly.csv.gz


## 7. Construct the national monthly modeling panel

Commodity prices have different units and price levels. Each commodity series is
therefore rebased to its own 2015 mean (=100), then averaged across observed
commodities. This avoids letting an intrinsically expensive commodity dominate
the national mandi index.

The index is joined to national food CPI, World Bank commodity prices, and mean
state climate. Observation counts and commodity counts are retained as coverage
indicators.


In [9]:
# First calculate weighted national prices within commodity and month.
national_commodity = monthly_prices.assign(
    weighted_price=lambda frame: (
        frame["modal_price_rs_per_quintal"] * frame["source_rows"]
    )
)
national_commodity = (
    national_commodity.groupby(
        ["date", "Commodity"],
        as_index=False,
        observed=True,
        sort=False,
    )
    .agg(
        weighted_price=("weighted_price", "sum"),
        source_rows=("source_rows", "sum"),
    )
)
national_commodity["national_price"] = (
    national_commodity["weighted_price"] / national_commodity["source_rows"]
)

# Use the 2015 mean as the common base. All eight assignment commodities have
# 2015 observations; the explicit check protects against silent missing bases.
base_2015 = (
    national_commodity.loc[national_commodity["date"].dt.year.eq(2015)]
    .groupby("Commodity", observed=True)["national_price"]
    .mean()
)
missing_bases = sorted(
    set(national_commodity["Commodity"].astype(str).unique()) - set(base_2015.index.astype(str))
)
if missing_bases:
    raise AssertionError(f"Missing 2015 base prices for: {missing_bases}")

# Mapping from a categorical key can preserve a categorical result in newer
# pandas versions. Convert explicitly to float before arithmetic for portability.
national_commodity["base_2015"] = (
    national_commodity["Commodity"].map(base_2015).astype("float64")
)
national_commodity["commodity_price_index_2015_100"] = (
    100 * national_commodity["national_price"] / national_commodity["base_2015"]
)

national_mandi = (
    national_commodity.groupby("date", as_index=False)
    .agg(
        mandi_price_index_2015_100=("commodity_price_index_2015_100", "mean"),
        mandi_source_rows=("source_rows", "sum"),
        commodities_observed=("Commodity", "nunique"),
    )
    .sort_values("date")
)

# This curated table already contains food CPI, World Bank prices, and national
# climate features. We add the independent AGMARKNET mandi index.
national_existing = pd.read_csv(
    CURATED / "india_food_affordability_national_monthly_2000_2026_ytd.csv.gz",
    parse_dates=["date"],
)
climate_national = (
    climate.groupby("date", as_index=False)
    .agg(
        state_avg_rainfall_mm=("rainfall_mm", "mean"),
        state_avg_temperature_c=("temperature_c", "mean"),
        reporting_regions=("region", "nunique"),
    )
)

national_modeling = (
    national_existing.merge(national_mandi, on="date", how="left")
    .merge(climate_national, on="date", how="left")
    .sort_values("date")
)

# pct_change uses only past values, preserving time direction for later models.
national_modeling["mandi_index_mom_pct"] = (
    national_modeling["mandi_price_index_2015_100"].pct_change() * 100
)
national_modeling["mandi_index_yoy_pct"] = (
    national_modeling["mandi_price_index_2015_100"].pct_change(12) * 100
)

national_path = PROCESSED / "modeling_monthly_national.csv.gz"
national_modeling.to_csv(national_path, index=False, compression="gzip")

print(f"National monthly rows: {len(national_modeling):,}")
print(f"Date range: {national_modeling['date'].min().date()} to {national_modeling['date'].max().date()}")
print(f"Rows with CPI: {national_modeling['food_cpi_2015_100'].notna().sum():,}")
print(f"Rows with mandi index: {national_modeling['mandi_price_index_2015_100'].notna().sum():,}")
print(f"Saved: {national_path}")


National monthly rows: 319
Date range: 2000-01-01 to 2026-07-01
Rows with CPI: 312
Rows with mandi index: 307
Saved: /content/drive/MyDrive/QM640_Food_Affordability/data/processed/modeling_monthly_national.csv.gz


## 8. Final quality summary and hand-off

The JSON summary below is the key result to share before Notebook 02 is created.
It records the exact processed counts, source total, temporal coverage, match
rate, and output locations.


## 8. Original-synopsis source readiness and data-gap audit

The original synopsis remains the controlling research design. This section
does **not** redefine a research question to match currently available data.
Instead, it distinguishes:

1. a confirmatory source that is available;
2. an interim proxy that may support preliminary analysis only; and
3. a missing source that must be acquired before the relevant null hypothesis
   can be finally tested.

Important interpretation rules:

- `source_rows` and `daily_records` measure reporting coverage; they are **not**
  market-arrival quantities.
- NASA POWER climate data are a reproducible interim climate source; they are
  not relabeled as IMD observations.
- FAOSTAT national annual production is not a substitute for state-commodity
  area, production, and yield.
- Published rural/urban HCES shares support a prototype HFASI but do not provide
  household-level or expenditure-fractile validity evidence.

The audit preserves the four original hypothesis pairs:

- **H0-1/H1-1:** joint food-price driver coefficients equal zero versus at
  least one nonzero coefficient.
- **H0-2/H1-2:** machine-learning forecast loss is not lower versus lower than
  the best statistical baseline.
- **H0-3/H1-3:** HFASI lacks versus demonstrates positive criterion validity
  and stable household-segment rankings.
- **H0-4/H1-4:** model-driven warnings do not reduce versus reduce paired
  retrospective decision loss.


In [10]:
# Store manually downloaded confirmatory files in this stable Drive folder.
EXTERNAL_REQUIRED = OUTPUT_ROOT / "data" / "external_required"
INPUT_TEMPLATES = OUTPUT_ROOT / "data" / "input_templates"
EXTERNAL_REQUIRED.mkdir(parents=True, exist_ok=True)
INPUT_TEMPLATES.mkdir(parents=True, exist_ok=True)

# The URLs are official discovery/download pages. Interactive portals may require
# a one-time manual export because their session tokens are not stable APIs.
required_sources = [
    {
        "source_id": "agmarknet_prices",
        "rq": "RQ1/RQ2",
        "required_variables": "date, state, market/region, commodity, modal price",
        "status_without_upload": "available_confirmatory",
        "current_evidence": "Curated official AGMARKNET price aggregate",
        "upload_patterns": [],
        "official_url": "https://agmarknet.gov.in/",
        "target_filename": "not_required_currently.csv",
        "critical": True,
    },
    {
        "source_id": "market_arrivals",
        "rq": "RQ1/RQ2",
        "required_variables": "date, state, market, commodity, arrival quantity, unit",
        "status_without_upload": "missing_confirmatory",
        "current_evidence": "No genuine arrival-quantity field; source_rows is coverage only",
        "upload_patterns": ["*arrival*.csv", "*arrival*.csv.gz", "*arrival*.xlsx"],
        "official_url": "https://enam.gov.in/web/dashboard/Historical",
        "target_filename": "agmarknet_enam_arrivals.csv.gz",
        "critical": True,
    },
    {
        "source_id": "state_crop_apy",
        "rq": "RQ1",
        "required_variables": "state, crop, year/season, area, production, yield",
        "status_without_upload": "provisional_proxy",
        "current_evidence": "Official DES state/district crop APY acquired and normalized",
        "upload_patterns": ["*apy*.csv", "*apy*.csv.gz", "*apy*.xlsx", "*production_yield*.csv*"],
        "official_url": "https://data.desagri.gov.in/website/apy-index-report-web",
        "target_filename": "des_state_crop_area_production_yield.csv.gz",
        "critical": True,
    },
    {
        "source_id": "rural_wages",
        "rq": "RQ1/RQ3",
        "required_variables": "year, month, state, occupation, male wage, female wage",
        "status_without_upload": "missing_confirmatory",
        "current_evidence": "Official Labour Bureau FY2025-26 portal export; partial historical coverage",
        "upload_patterns": ["*rural*wage*.csv", "*rural*wage*.csv.gz", "*rural*wage*.xlsx"],
        "official_url": "https://www.labourbureau.gov.in/rural-wages",
        "target_filename": "labour_bureau_rural_wages_monthly.csv.gz",
        "critical": True,
    },
    {
        "source_id": "hces_microdata",
        "rq": "RQ3",
        "required_variables": "household key, sector, state, survey weight, food expenditure, MPCE",
        "status_without_upload": "missing_confirmatory",
        "current_evidence": "HCES 2022-23/2023-24 microdata acquired locally; disclosure-safe segment aggregate prepared",
        "upload_patterns": ["*hces*.csv", "*hces*.csv.gz", "*hces*.dta", "*hces*.sav", "*hces*.zip"],
        "official_url": "https://microdata.gov.in/NADA/index.php/catalog/224",
        "target_filename": "hces_2022_23_unit_level_files.zip",
        "critical": True,
    },
    {
        "source_id": "mospi_cpi_cfpi",
        "rq": "RQ1/RQ2/RQ3",
        "required_variables": "month, CPI/CFPI index, group, sector/state where available",
        "status_without_upload": "provisional_proxy",
        "current_evidence": "National food-CPI series available; official MOSPI confirmation preferred",
        "upload_patterns": ["*mospi*cpi*.csv", "*mospi*cpi*.xlsx", "*cfpi*.csv", "*cfpi*.xlsx"],
        "official_url": "https://cpi.mospi.gov.in/",
        "target_filename": "mospi_cpi_cfpi_monthly.csv.gz",
        "critical": False,
    },
    {
        "source_id": "climate",
        "rq": "RQ1",
        "required_variables": "date, state, rainfall, temperature, humidity",
        "status_without_upload": "available_interim_proxy",
        "current_evidence": "NASA POWER state/UT climate panel",
        "upload_patterns": ["*imd*rain*.csv", "*imd*climate*.csv", "*imd*.xlsx"],
        "official_url": "https://mausam.imd.gov.in/",
        "target_filename": "imd_state_monthly_climate.csv.gz",
        "critical": False,
    },
    {
        "source_id": "global_food_energy",
        "rq": "RQ1/RQ2",
        "required_variables": "month, wheat/rice/oil/sugar prices, crude-oil price",
        "status_without_upload": "available_confirmatory",
        "current_evidence": "World Bank Pink Sheet monthly indicators",
        "upload_patterns": [],
        "official_url": "https://www.worldbank.org/en/research/commodity-markets",
        "target_filename": "not_required_currently.csv",
        "critical": False,
    },
]


def uploaded_matches(patterns):
    # Return matching user-supplied files without reading large microdata.
    matches = []
    for pattern in patterns:
        matches.extend(EXTERNAL_REQUIRED.rglob(pattern))
    return sorted({str(path) for path in matches if path.is_file()})


audit_rows = []
for item in required_sources:
    matches = uploaded_matches(item["upload_patterns"])
    if matches and item["source_id"] == "rural_wages":
        status = "partial_confirmatory_coverage"
    elif matches and item["source_id"] == "hces_microdata":
        status = "available_confirmatory_aggregate"
    else:
        status = "available_confirmatory_upload" if matches else item["status_without_upload"]
    audit_rows.append({
        "source_id": item["source_id"],
        "research_question": item["rq"],
        "required_variables": item["required_variables"],
        "status": status,
        "critical_for_final_hypothesis": item["critical"],
        "current_evidence": item["current_evidence"],
        "uploaded_files": " | ".join(matches),
        "official_url": item["official_url"],
        "target_filename": item["target_filename"],
    })

data_gap_audit = pd.DataFrame(audit_rows)
data_gap_audit.to_csv(REPORT_OUTPUT / "01_required_data_audit.csv", index=False)

# Header-only templates make the expected schema explicit without inventing data.
templates = {
    "agmarknet_enam_arrivals_template.csv": [
        "date", "state", "district", "market", "commodity",
        "arrival_quantity", "arrival_unit", "source_url",
    ],
    "des_state_crop_apy_template.csv": [
        "state", "crop", "season", "year", "area_hectare",
        "production_tonne", "yield_kg_per_hectare", "source_url",
    ],
    "labour_bureau_rural_wages_template.csv": [
        "year", "month", "state", "occupation_group", "occupation_item",
        "male_wage_rs_per_day", "female_wage_rs_per_day", "source_url",
    ],
    "hces_household_analysis_template.csv": [
        "household_id", "state", "sector", "expenditure_fractile",
        "household_size", "food_expenditure_monthly", "mpce",
        "survey_weight", "source_file",
    ],
    "mospi_cpi_cfpi_template.csv": [
        "date", "sector", "state", "group", "subgroup",
        "index_base", "index_value", "source_url",
    ],
}
for filename, columns in templates.items():
    pd.DataFrame(columns=columns).to_csv(INPUT_TEMPLATES / filename, index=False)

# This matrix links every original hypothesis to at least four measurable variables.
hypothesis_variable_coverage = pd.DataFrame([
    ["RQ1", "future_price_change_h1_h3", "lagged_price", "available", "confirmatory after Notebook 02"],
    ["RQ1", "future_price_change_h1_h3", "market_arrivals", "missing", "download required"],
    ["RQ1", "future_price_change_h1_h3", "rainfall_temperature_anomaly", "available_proxy", "NASA POWER interim"],
    ["RQ1", "future_price_change_h1_h3", "production_yield", "available", "official DES state-crop APY"],
    ["RQ1", "future_price_change_h1_h3", "fuel_logistics_cost", "available", "World Bank crude oil"],
    ["RQ1", "future_price_change_h1_h3", "wage_macro_growth", "partial", "Labour Bureau FY2025-26 only"],
    ["RQ2", "forecast_loss", "model_family", "derived_later", "Notebook 05"],
    ["RQ2", "forecast_loss", "forecast_horizon", "derived_later", "1, 2, 3 months"],
    ["RQ2", "forecast_loss", "commodity_region", "available", "state panel keys"],
    ["RQ2", "forecast_loss", "forecast_origin", "derived_later", "rolling validation"],
    ["RQ3", "hfasi_validity", "forecast_food_cost_growth", "derived_later", "Notebook 05/08"],
    ["RQ3", "hfasi_validity", "food_expenditure_share", "available_aggregate", "HCES-derived state/sector/decile"],
    ["RQ3", "hfasi_validity", "wage_income_growth", "partial", "Labour Bureau FY2025-26 plus PLFS"],
    ["RQ3", "hfasi_validity", "household_segment_fractile", "available_aggregate", "HCES expenditure deciles"],
    ["RQ4", "decision_loss_gain", "shock_probability", "derived_later", "Notebook 06"],
    ["RQ4", "decision_loss_gain", "hfasi", "derived_later", "Notebook 08"],
    ["RQ4", "decision_loss_gain", "warning_strategy", "derived_later", "Notebook 09"],
    ["RQ4", "decision_loss_gain", "scenario_severity_lead_time", "derived_later", "Notebook 09"],
], columns=["research_question", "outcome", "variable", "status", "action"])
hypothesis_variable_coverage.to_csv(
    REPORT_OUTPUT / "01_hypothesis_variable_coverage.csv", index=False
)

critical_gap_mask = (
    data_gap_audit["critical_for_final_hypothesis"]
    & ~data_gap_audit["status"].str.startswith("available_confirmatory")
)
critical_data_gaps = data_gap_audit.loc[
    critical_gap_mask, ["source_id", "status", "official_url", "target_filename"]
].copy()

print("ORIGINAL-SYNOPSIS DATA READINESS")
print(data_gap_audit[
    ["source_id", "research_question", "status", "critical_for_final_hypothesis"]
].to_string(index=False))

if len(critical_data_gaps):
    print("\nACTION REQUIRED FOR FINAL CONFIRMATORY TESTS")
    print(critical_data_gaps.to_string(index=False))
    print(f"\nUpload downloaded files to: {EXTERNAL_REQUIRED}")
else:
    print("\nAll critical confirmatory source classes are present.")


ORIGINAL-SYNOPSIS DATA READINESS
         source_id research_question                           status  critical_for_final_hypothesis
  agmarknet_prices           RQ1/RQ2           available_confirmatory                           True
   market_arrivals           RQ1/RQ2             missing_confirmatory                           True
    state_crop_apy               RQ1    available_confirmatory_upload                           True
       rural_wages           RQ1/RQ3    partial_confirmatory_coverage                           True
    hces_microdata               RQ3 available_confirmatory_aggregate                           True
    mospi_cpi_cfpi       RQ1/RQ2/RQ3                provisional_proxy                          False
           climate               RQ1          available_interim_proxy                          False
global_food_energy           RQ1/RQ2           available_confirmatory                          False

ACTION REQUIRED FOR FINAL CONFIRMATORY TESTS
      source

In [11]:
summary = {
    "notebook": "01_data_acquisition_synopsis_aligned",
    "status": (
        "completed_with_confirmatory_data_gaps"
        if len(critical_data_gaps)
        else "completed_confirmatory_sources_present"
    ),
    "original_research_questions_preserved": True,
    "hypothesis_variable_groups_audited": 4,
    "required_source_classes_audited": int(len(data_gap_audit)),
    "critical_confirmatory_data_gaps": int(len(critical_data_gaps)),
    "critical_gap_sources": critical_data_gaps["source_id"].tolist(),
    "curated_files_audited": int(len(inventory)),
    "curated_analytical_rows": int(inventory["rows"].sum()),
    "agmarknet_daily_aggregate_rows": int(daily_aggregate_rows),
    "agmarknet_underlying_official_observations": int(underlying_source_rows),
    "agmarknet_monthly_state_commodity_rows": int(len(monthly_prices)),
    "agmarknet_states_uts": int(monthly_prices["region"].nunique()),
    "agmarknet_commodities": int(monthly_prices["Commodity"].nunique()),
    "agmarknet_start": str(monthly_prices["date"].min().date()),
    "agmarknet_end": str(monthly_prices["date"].max().date()),
    "state_modeling_rows": int(len(state_modeling_panel)),
    "climate_match_rate": float(climate_match_rate),
    "national_modeling_rows": int(len(national_modeling)),
    "national_start": str(national_modeling["date"].min().date()),
    "national_end": str(national_modeling["date"].max().date()),
    "source_rows_is_not_market_arrivals": True,
    "hces_aggregate_is_not_household_validation": True,
    "partial_2026_warning": True,
    "climate_spatial_limitation": (
        "Representative administrative-capital points; not area-weighted state averages."
    ),
    "external_required_folder": str(EXTERNAL_REQUIRED),
    "output_root": str(OUTPUT_ROOT),
}

summary_path = REPORT_OUTPUT / "01_execution_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("=" * 78)
print("NOTEBOOK 01 ALIGNED EXECUTION SUMMARY - PLEASE SHARE THIS OUTPUT")
print("=" * 78)
print(json.dumps(summary, indent=2))
print("\nGenerated files:")
generated_paths = [
    PROCESSED / "source_inventory.csv",
    monthly_price_path,
    state_panel_path,
    national_path,
    REPORT_OUTPUT / "01_commodity_coverage.csv",
    REPORT_OUTPUT / "01_required_data_audit.csv",
    REPORT_OUTPUT / "01_hypothesis_variable_coverage.csv",
    summary_path,
]
for path in generated_paths:
    print(f"- {path} ({path.stat().st_size / 1_000_000:.3f} MB)")
print(f"- {INPUT_TEMPLATES} ({len(templates)} schema templates)")


NOTEBOOK 01 ALIGNED EXECUTION SUMMARY - PLEASE SHARE THIS OUTPUT
{
  "notebook": "01_data_acquisition_synopsis_aligned",
  "status": "completed_with_confirmatory_data_gaps",
  "original_research_questions_preserved": true,
  "hypothesis_variable_groups_audited": 4,
  "required_source_classes_audited": 8,
  "critical_confirmatory_data_gaps": 2,
  "critical_gap_sources": [
    "market_arrivals",
    "rural_wages"
  ],
  "curated_files_audited": 14,
  "curated_analytical_rows": 1494745,
  "agmarknet_daily_aggregate_rows": 969377,
  "agmarknet_underlying_official_observations": 18836462,
  "agmarknet_monthly_state_commodity_rows": 41792,
  "agmarknet_states_uts": 32,
  "agmarknet_commodities": 8,
  "agmarknet_start": "2001-01-01",
  "agmarknet_end": "2026-07-01",
  "state_modeling_rows": 41792,
  "climate_match_rate": 1.0,
  "national_modeling_rows": 319,
  "national_start": "2000-01-01",
  "national_end": "2026-07-01",
  "source_rows_is_not_market_arrivals": true,
  "hces_aggregate_is_not

## Notebook 01 conclusion and hand-off

Notebook 01 has created the existing modeling panels and audited the additional
source classes required by the original synopsis. A status of
`completed_with_confirmatory_data_gaps` is not a notebook failure: it means the
current data support interim analysis while one or more final confirmatory
variables still require an official download.

Before Notebook 02:

1. share the printed execution-summary block;
2. if possible, download the files listed under **ACTION REQUIRED**;
3. place them in the printed `external_required` Google Drive folder; and
4. do not rename reporting-coverage fields as market arrivals.
